<a href="https://colab.research.google.com/github/Adhira-Deogade/pytorch-learnings/blob/main/creating_dataloaders.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Docs
# https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html
# Dataset stores the samples and their corresponding labels
# DataLoader wraps an iterable around the Dataset to enable easy access to the samples.

In [2]:
# Creating a Custom Dataset for your files

In [3]:
# The Dataset retrieves our dataset’s features and labels one sample at a time.
# While training a model, we typically want to pass samples in “minibatches”,
# reshuffle the data at every epoch to reduce model overfitting,
# and use Python’s multiprocessing to speed up data retrieval.

In [4]:
# torch.squeeze() eliminates dimensions with a size of 1, effectively reducing the tensor's dimensionality.


In [9]:
import torch
import numpy as np

In [5]:


# Example tensor with dimensions of size 1
x = torch.zeros(2, 1, 2, 1, 2)
print("Original shape:", x.shape)  # Output: torch.Size([2, 1, 2, 1, 2])

# Squeeze all dimensions of size 1
y = torch.squeeze(x)
print("Squeezed shape:", y.shape)  # Output: torch.Size([2, 2, 2])

# Squeeze specific dimension (dimension 1)
z = torch.squeeze(x, 1)
print("Squeezed with dim=1:", z.shape)  # Output: torch.Size([2, 2, 1, 2])


Original shape: torch.Size([2, 1, 2, 1, 2])
Squeezed shape: torch.Size([2, 2, 2])
Squeezed with dim=1: torch.Size([2, 2, 1, 2])


In [6]:
# Goals:
# 1. Create a numbers only Dataset/ Dataloader
# 2. Create an images Dataset/Dataloader

In [8]:
# Create a dataset using CoffeeRoasting dataset

In [10]:
def load_coffee_data():
    rng = np.random.default_rng(2)
    X = rng.random(400).reshape(-1,2)
    X[:,1] = X[:,1] * 4 + 11.5          # 12-15 min is best
    X[:,0] = X[:,0] * (285-150) + 150  # 350-500 F (175-260 C) is best
    Y = np.zeros(len(X))

    i=0
    for t,d in X:
        y = -3/(260-175)*t + 21
        if (t > 175 and t < 260 and d > 12 and d < 15 and d<=y ):
            Y[i] = 1
        else:
            Y[i] = 0
        i += 1

    return (X, Y.reshape(-1,1))

In [11]:
# The above has been sourced from Andre Ng's course

In [12]:
X, Y = load_coffee_data()
print(X.shape)
print(Y.shape)


(200, 2)
(200, 1)


In [13]:
# These are numpy arrays, let's convert them to Tensors

In [15]:
X_tensor = torch.from_numpy(X)
Y_tensor = torch.from_numpy(Y)
print(X_tensor.shape)
print(Y_tensor.shape)

torch.Size([200, 2])
torch.Size([200, 1])


In [17]:
# Let's scale them between 0 and 1 using scikit-learn's MinMaxScaler()
# For that we need, Min and Max value of each tensor

In [41]:
X_min = X_tensor.min(dim=0).values # Column-wise min
X_max = X_tensor.max(dim=0).values # Column-wise max
print(X_min)
print(X_max)

tensor([151.3237,  11.5127], dtype=torch.float64)
tensor([284.9943,  15.4542], dtype=torch.float64)


In [42]:
Y_min = Y_tensor.min(dim=0).values # Column-wise min
Y_max = Y_tensor.max(dim=0).values # Column-wise max
print(Y_min)
print(Y_max)

tensor([0.], dtype=torch.float64)
tensor([1.], dtype=torch.float64)


In [43]:
# X_scaled = (X - X_min) / (X_max - X_min)

In [44]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Sample data
data = np.array([[1, 2],
                 [3, 4],
                 [5, 6]])
# print(data.shape)
# print(data)
# Initialize MinMaxScaler
scaler = MinMaxScaler()

# Fit and transform the data
scaled_data = scaler.fit(data)
dmax = scaled_data.data_max_
dmin = scaled_data.data_min_

print(f"Dmax: {dmax}")
print(f"Dmin: {dmin}")
dsclaled = (data-dmin)/(dmax-dmin)
print(dsclaled)
# print(scaler.fit_transform(data))

Dmax: [5. 6.]
Dmin: [1. 2.]
[[0.  0. ]
 [0.5 0.5]
 [1.  1. ]]


In [45]:
# Okay, I see,
# Subtract the entire column with min and divide by max-min

In [46]:
X_max_minus_min = X_max - X_min
print(X_max_minus_min)


tensor([133.6706,   3.9415], dtype=torch.float64)


In [47]:
Y_max_minus_min = Y_max - Y_min
print(Y_max_minus_min)

tensor([1.], dtype=torch.float64)


In [48]:
X_scaled = (X_tensor - X_min) / (X_max_minus_min)
Y_scaled = (Y_tensor - Y_min) / (Y_max_minus_min)
print(X_scaled[:3])
print(Y_scaled[:3])

tensor([[0.2543, 0.2997],
        [0.8124, 0.0900],
        [0.5962, 0.7361]], dtype=torch.float64)
tensor([[1.],
        [0.],
        [0.]], dtype=torch.float64)


In [50]:
print(torch.max(X_scaled))
print(torch.max(Y_scaled))

tensor(1., dtype=torch.float64)
tensor(1., dtype=torch.float64)


In [51]:
print(torch.min(X_scaled))
print(torch.min(Y_scaled))

tensor(0., dtype=torch.float64)
tensor(0., dtype=torch.float64)


In [52]:
# Let's tile the dataset to 10,000 values

In [54]:
print(Y[:10])

[[1.]
 [0.]
 [0.]
 [0.]
 [1.]
 [1.]
 [0.]
 [0.]
 [0.]
 [1.]]


In [55]:
# This is a binary classfication
# The values of Y are either 1 or 0, so I didn't need a normalization on it

In [57]:
# Repeat the data 1000 times along the first dimension and 1 time along the
# second dimension
X_tile = torch.tile(X_scaled, (1000, 1))
Y_tile = torch.tile(Y_scaled, (1000, 1))
print(X_tile.shape)
print(Y_tile.shape)

torch.Size([200000, 2])
torch.Size([200000, 1])


In [58]:
# Example
import torch

# Define a tensor
ten = torch.tensor([13, 24, 35, 46])

# Repeat the tensor 2 times along the first dimension and 3 times along the second dimension
res = torch.tile(ten, (2, 3))

# Print the result
print(res)


tensor([[13, 24, 35, 46, 13, 24, 35, 46, 13, 24, 35, 46],
        [13, 24, 35, 46, 13, 24, 35, 46, 13, 24, 35, 46]])


In [62]:
# Let's create a Dataloader now
# First let's create a Dataset and then DataLoader will be an
# iterator over this Dataset

In [63]:
from torch.utils.data import Dataset
class CustomCoffeeDataLoader(Dataset):
  def __init__(self, X_input, Y_input):
    self.X_output = X_input
    self.Y_output = Y_input
  def __len__(self):
    return len(self.Y_output)
  def __getitem__(self, idx):
    return self.X_output[idx], self.Y_output[idx]


In [81]:
# Whoops we need training and testing data separately
# Let's use scikit-learns train-test splitting feature

In [82]:
from sklearn.model_selection import train_test_split

In [83]:
X_train, X_test, Y_train, Y_test = train_test_split(X_tile, Y_tile, test_size=0.2, random_state=42)
print(X_train.shape)
print(Y_train.shape)
print(X_test.shape)
print(Y_test.shape)

torch.Size([160000, 2])
torch.Size([160000, 1])
torch.Size([40000, 2])
torch.Size([40000, 1])


In [84]:
CoffeeDatasetTrain = CustomCoffeeDataLoader(X_train, Y_train)
CoffeeDatasetTest = CustomCoffeeDataLoader(X_test, Y_test)
print(CoffeeDatasetTrain, CoffeeDatasetTest)

<__main__.CustomCoffeeDataLoader object at 0x7ea175533b50> <__main__.CustomCoffeeDataLoader object at 0x7ea175504410>


In [85]:
print(len(CoffeeDatasetTrain), len(CoffeeDatasetTest))

160000 40000


In [87]:
print(CoffeeDatasetTrain[0])
print(CoffeeDatasetTest[-1])

(tensor([0.5188, 0.0037], dtype=torch.float64), tensor([0.], dtype=torch.float64))
(tensor([0.9198, 0.2560], dtype=torch.float64), tensor([0.], dtype=torch.float64))


In [88]:
# Let's create an iterator/ DataLoader for this
# We need a batch size, let's set it to 64

In [89]:
from torch.utils.data import DataLoader
coffee_batch_size = 64
CoffeeDataLoaderTrain = DataLoader(CoffeeDatasetTrain, batch_size=coffee_batch_size, shuffle=True)
CoffeeDataLoaderTest = DataLoader(CoffeeDatasetTest, batch_size=coffee_batch_size, shuffle=True)

In [90]:
print(CoffeeDataLoaderTrain)
print(CoffeeDataLoaderTest)

In [91]:
print(len(CoffeeDataLoaderTrain))
print(len(CoffeeDataLoaderTest))

2500
625


In [95]:
# print(next(iter(CoffeeDataLoaderTrain)))

In [98]:
for batch, (X_batch, Y_batch) in enumerate(CoffeeDataLoaderTest):
  print(f"Batch: {batch+1}")
  print(f"X_batch shape: {X_batch.shape}")
  print(f"Y_batch shape: {Y_batch.shape}")
  break

Batch: 1
X_batch shape: torch.Size([64, 2])
Y_batch shape: torch.Size([64, 1])
